# 🏎️ F1 Race Position Predictor — Exploratory Data Analysis

This notebook explores the processed feature dataset to understand patterns,
correlations, and feature distributions before model training.

In [ ]:
from __future__ import annotations

import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns

from f1_predictor.config import settings
from f1_predictor.features.engineering import FEATURE_COLUMNS, TARGET_COLUMN

pd.set_option('display.max_columns', 30)
print('Imports loaded ✓')

In [ ]:
# Load processed features
df = pd.read_csv(settings.project_root / settings.data_processed_dir / 'features.csv')
print(f'Dataset: {len(df)} rows, {len(df.columns)} columns')
print(f'Features: {len(FEATURE_COLUMNS)}')
print(f'\nColumns: {list(df.columns)}')
df.head()

## 🎯 Grid Position vs Finishing Position

The baseline assumption: drivers finish where they start. How true is this?

In [ ]:
fig = px.scatter(df, x='grid_position', y='finishing_position',
                 color='team_name' if 'team_name' in df.columns else None,
                 hover_data=['full_name'] if 'full_name' in df.columns else None,
                 opacity=0.5,
                 title='Grid Position vs Finishing Position')
fig.add_shape(type='line', x0=0, y0=0, x1=20, y1=20,
              line=dict(dash='dash', color='gray'))
fig.update_layout(height=600)
fig.show()

from sklearn.metrics import mean_absolute_error
baseline_mae = mean_absolute_error(df['finishing_position'], df['grid_position'])
print(f'Baseline MAE (grid = finish): {baseline_mae:.2f} positions')

## 📊 Feature Correlation Heatmap

Which features are most correlated with finishing position?

In [ ]:
corr_cols = FEATURE_COLUMNS + [TARGET_COLUMN]
corr_cols = [c for c in corr_cols if c in df.columns]
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(16, 12))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, ax=ax, linewidths=0.5)
ax.set_title('Feature Correlation Matrix', fontsize=16)
plt.tight_layout()
plt.show()

# Top correlations with target
target_corr = corr[TARGET_COLUMN].drop(TARGET_COLUMN).sort_values(key=abs, ascending=False)
print('\nCorrelation with finishing_position:')
for feat, val in target_corr.items():
    print(f'  {feat:30s} {val:+.3f}')

## 🌧️ Weather Impact: Wet vs Dry Races

Do wet races produce more unpredictable results?

In [ ]:
if 'is_wet_race' in df.columns:
    df['position_change'] = df['grid_position'] - df['finishing_position']
    
    wet = df[df['is_wet_race'] == 1]['position_change']
    dry = df[df['is_wet_race'] == 0]['position_change']
    
    print(f'Dry races:  mean position change = {dry.mean():+.2f}, std = {dry.std():.2f} ({len(dry)} entries)')
    print(f'Wet races:  mean position change = {wet.mean():+.2f}, std = {wet.std():.2f} ({len(wet)} entries)')
    if dry.std() > 0:
        print(f'\nWet races are {wet.std()/dry.std():.1f}x more unpredictable')
    
    fig = px.histogram(df, x='position_change', color='is_wet_race',
                       barmode='overlay', nbins=30,
                       color_discrete_map={0: '#16213e', 1: '#E10600'},
                       labels={'is_wet_race': 'Wet Race'},
                       title='Position Changes: Wet vs Dry Races')
    fig.show()
else:
    print('No weather data available')

## 🏙️ Street Circuit Impact

Are street circuits more unpredictable than permanent tracks?

In [ ]:
if 'is_street_circuit' in df.columns:
    df['abs_position_change'] = abs(df['grid_position'] - df['finishing_position'])
    
    street = df[df['is_street_circuit'] == 1]['abs_position_change']
    perm = df[df['is_street_circuit'] == 0]['abs_position_change']
    
    print(f'Permanent circuits: mean |pos change| = {perm.mean():.2f}')
    print(f'Street circuits:    mean |pos change| = {street.mean():.2f}')
    
    if 'circuit_short_name' in df.columns:
        circuit_volatility = (
            df.groupby('circuit_short_name')['abs_position_change']
            .agg(['mean', 'std', 'count'])
            .sort_values('mean', ascending=False)
        )
        print('\nPosition change volatility by circuit:')
        print(circuit_volatility.to_string())
else:
    print('No circuit type data available')